# Generate MultiSocial Micro Dataset (Colab)

This notebook creates `multisocial_micro.csv` from your own uploaded CSV file using a single generation provider: Groq.

## Required packages

- pandas
- tqdm
- groq
- scikit-learn

## Expected input

Upload a headerless CSV using this schema:
`["target", "ids", "date", "flag", "user", "text"]`

## Before running

Set the Groq API key in Colab environment variables (never hardcode keys in notebook cells):

- `GROQ_API_KEY`

## Execution

This notebook processes all sampled rows with Groq (`llama-3.1-8b-instant`).


In [ ]:
# Install dependencies (Colab)
!pip -q install pandas tqdm groq scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.2 MB/s eta 0:00:00


In [ ]:
import os
import time
from typing import List, Tuple

import pandas as pd
from groq import Groq
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -----------------------------
# Global constants
# -----------------------------
RANDOM_SEED = 42
HUMAN_SAMPLE_SIZE = 2000
GROQ_MODEL_NAME = "llama-3.1-8b-instant"
OUTPUT_CSV_PATH = "multisocial_micro.csv"
MAX_RETRIES = 5
INITIAL_BACKOFF_SECONDS = 4
SUCCESS_CALL_SLEEP_SECONDS = 2.5  # 30 RPM safety margin

# IMPORTANT: set API key before running main().
# os.environ["GROQ_API_KEY"] = "..."

In [ ]:
# Optional tuning cell for quotas and speed
GROQ_MODEL_NAME = "llama-3.1-8b-instant"
MAX_RETRIES = 5
INITIAL_BACKOFF_SECONDS = 4
SUCCESS_CALL_SLEEP_SECONDS = 2.5

print(f"Groq model: {GROQ_MODEL_NAME}")
print(f"Retries: {MAX_RETRIES} | initial backoff: {INITIAL_BACKOFF_SECONDS}s | success sleep: {SUCCESS_CALL_SLEEP_SECONDS}s")

Groq model: llama-3.1-8b-instant
Retries: 5 | initial backoff: 4s | success sleep: 2.5s


In [ ]:
def fetch_human_data(input_csv_path: str, sample_size: int = HUMAN_SAMPLE_SIZE) -> pd.DataFrame:
    """
    Stage 1: Load and sample human-written social media texts from a headerless CSV.

    Expected CSV structure (no header row):
    ["target", "ids", "date", "flag", "user", "text"]
    """
    expected_columns = ["target", "ids", "date", "flag", "user", "text"]
    print(f"Loading user dataset from: {input_csv_path}")
    raw_df = pd.read_csv(input_csv_path, header=None, names=expected_columns, encoding="ISO-8859-1")

    if raw_df.empty:
        raise ValueError("Uploaded CSV is empty.")

    text_series = raw_df["text"].astype("string").fillna("").str.strip()
    text_series = text_series[text_series != ""]

    if len(text_series) < sample_size:
        raise ValueError(
            f"Requested sample_size={sample_size}, but only {len(text_series)} non-empty texts are available."
        )

    sampled_texts = text_series.sample(n=sample_size, random_state=RANDOM_SEED, replace=False).tolist()

    human_df = pd.DataFrame({
        "text": pd.Series(sampled_texts, dtype="string").fillna(""),
        "label": 0,
        "multi_label": "human",
        "source": "twitter",
    })

    print(f"Assigned columns: {expected_columns}")
    print(f"Human dataset prepared with {len(human_df)} rows.")
    return human_df


def configure_groq_client() -> Groq:
    """Configure Groq client from environment variable."""
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("Missing GROQ_API_KEY environment variable.")
    return Groq(api_key=api_key)


def build_paraphrase_prompt(text: str) -> str:
    """Create the exact prompt used for paraphrasing."""
    return (
        "Paraphrase the following social media post. Keep the exact same informal tone, "
        "length, hashtags, and emojis. Do not add any conversational filler. "
        "Just output the paraphrased text. Text: "
        f"{text}"
    )


def _is_rate_limited_exception(exc: Exception) -> bool:
    """Best-effort 429 detection for SDK exceptions."""
    code = getattr(exc, "status_code", None)
    if code is None:
        code = getattr(exc, "code", None)
    if code == 429:
        return True
    msg = str(exc).lower()
    return "429" in msg or ("rate" in msg and "limit" in msg)


def _call_groq(groq_client: Groq, prompt: str) -> Tuple[str, str]:
    """Call Groq and return (text, status). status: ok | rate_limited | failed."""
    try:
        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=280,
        )
        text = (completion.choices[0].message.content or "").strip()
        if text:
            return text, "ok"
        return "", "failed"
    except Exception as e:
        if _is_rate_limited_exception(e):
            print("Groq rate-limited (429).")
            return "", "rate_limited"
        print(f"Groq error: {e}")
        return "", "failed"


def generate_one_paraphrase(groq_client: Groq, original_text: str) -> Tuple[str, bool]:
    """
    Generate a single paraphrase using Groq only.

    - If 429 happens, apply exponential backoff and retry.
    - If all retries fail, return "", False.
    """
    prompt = build_paraphrase_prompt(original_text)
    backoff_seconds = INITIAL_BACKOFF_SECONDS

    for attempt in range(1, MAX_RETRIES + 1):
        generated_text, status = _call_groq(groq_client, prompt)

        if status == "ok" and generated_text.strip():
            return generated_text.strip(), True

        if status == "rate_limited" and attempt < MAX_RETRIES:
            print(
                f"Groq attempt {attempt}/{MAX_RETRIES} hit rate limit. "
                f"Sleeping {backoff_seconds}s before retry..."
            )
            time.sleep(backoff_seconds)
            backoff_seconds *= 2
            continue

        if attempt < MAX_RETRIES:
            print(
                f"Groq attempt {attempt}/{MAX_RETRIES} failed (status={status}). "
                f"Sleeping {backoff_seconds}s before retry..."
            )
            time.sleep(backoff_seconds)
            backoff_seconds *= 2

    return "", False


def generate_machine_data(human_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generate machine-written counterparts using Groq only.

    Keeps ONLY successful paraphrases to preserve a strict 1:1 balance between
    human and machine rows.
    """
    groq_client = configure_groq_client()

    valid_human_texts: List[str] = []
    generated_texts: List[str] = []
    success_count = 0
    dropped_count = 0

    print(f"Starting GROQ generation for {len(human_df)} rows...")

    for text in tqdm(human_df["text"].tolist(), desc="Generating (groq)", unit="post"):
        original_text = str(text)
        generated_text, success = generate_one_paraphrase(groq_client, original_text)

        if success:
            valid_human_texts.append(original_text)
            generated_texts.append(generated_text)
            success_count += 1
            # 2.5s sleep prevents exceeding 30 requests/minute on steady state.
            time.sleep(SUCCESS_CALL_SLEEP_SECONDS)
        else:
            dropped_count += 1

    if success_count == 0:
        raise RuntimeError(
            "No successful paraphrases were generated with Groq. "
            "Check GROQ_API_KEY/quota and try smaller HUMAN_SAMPLE_SIZE."
        )

    new_human_df = pd.DataFrame({
        "text": pd.Series(valid_human_texts, dtype="string").fillna(""),
        "label": 0,
        "multi_label": "human",
        "source": "twitter",
    })

    machine_df = pd.DataFrame({
        "text": pd.Series(generated_texts, dtype="string").fillna(""),
        "label": 1,
        "multi_label": GROQ_MODEL_NAME,
        "source": "twitter",
    })

    print(
        f"GROQ successful pairs: {success_count} | dropped rows: {dropped_count} | "
        f"machine label: {GROQ_MODEL_NAME}"
    )
    return new_human_df, machine_df


def add_split_with_stratification(df: pd.DataFrame) -> pd.DataFrame:
    """Create an 80/20 train/test split stratified by label for class balance."""
    _, test_idx = train_test_split(
        df.index,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=df["label"],
        shuffle=True,
    )

    df = df.copy()
    df["split"] = "train"
    df.loc[test_idx, "split"] = "test"
    return df


def format_and_save(human_df: pd.DataFrame, machine_df: pd.DataFrame, output_path: str) -> pd.DataFrame:
    """Stage 3: Concatenate, enrich, format, and save final dataset."""
    if len(human_df) != len(machine_df):
        raise RuntimeError("Human and machine datasets must have equal row counts.")

    combined_df = pd.concat([human_df, machine_df], ignore_index=True)
    combined_df["text"] = combined_df["text"].astype(str)
    combined_df["language"] = "en"
    combined_df["length"] = combined_df["text"].apply(lambda x: len(x.split()))
    combined_df["potential_noise"] = 0
    combined_df = add_split_with_stratification(combined_df)

    final_columns = [
        "text",
        "label",
        "multi_label",
        "split",
        "language",
        "length",
        "source",
        "potential_noise",
    ]
    final_df = combined_df[final_columns]

    final_df.to_csv(output_path, index=False)
    print(f"Saved dataset to {output_path}")
    print(f"Final rows: {len(final_df)} | Human: {len(human_df)} | Machine: {len(machine_df)}")
    return final_df


def main(input_csv_path: str) -> pd.DataFrame:
    """Run all stages with Groq only."""
    print("Pipeline started (Groq-only).")
    human_df = fetch_human_data(input_csv_path=input_csv_path, sample_size=HUMAN_SAMPLE_SIZE)
    provider_human_df, provider_machine_df = generate_machine_data(human_df)
    final_df = format_and_save(provider_human_df, provider_machine_df, output_path=OUTPUT_CSV_PATH)
    print("Pipeline completed (Groq-only).")
    return final_df

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
input_csv_path = '/content/drive/MyDrive/training.1600000.processed.noemoticon.csv'

if not os.path.exists(input_csv_path):
    raise FileNotFoundError(f"Could not find the file at {input_csv_path}. Please check the path.")

print(f"Using file from Drive: {input_csv_path}")

Mounted at /content/drive
Using file from Drive: /content/drive/MyDrive/training.1600000.processed.noemoticon.csv


In [ ]:
# Run Groq-only pipeline on all sampled rows
final_df = main(input_csv_path=input_csv_path)
print("Completed Groq-only run")
display(final_df.head())

Pipeline started (Groq-only).
Loading user dataset from: /content/drive/MyDrive/training.1600000.processed.noemoticon.csv
Assigned columns: ['target', 'ids', 'date', 'flag', 'user', 'text']
Human dataset prepared with 2000 rows.
Starting GROQ generation for 2000 rows...


Generating (groq): 100%|██████████| 2000/2000 [1:32:41<00:00,  2.78s/post]

GROQ successful pairs: 2000 | dropped rows: 0 | machine label: llama-3.1-8b-instant
Saved dataset to multisocial_micro.csv
Final rows: 4000 | Human: 2000 | Machine: 2000
Pipeline completed (Groq-only).
Completed Groq-only run


,text,label,multi_label,split,language,length,source,potential_noise
0,@chrishasboobs AHHH I HOPE YOUR OK!!!,0,human,train,en,6,twitter,0
1,"@misstoriblack cool , i have no tweet apps fo...",0,human,test,en,12,twitter,0
2,@TiannaChaos i know just family drama. its la...,0,human,train,en,27,twitter,0
3,School email won't open and I have geography ...,0,human,train,en,16,twitter,0
4,upper airways problem,0,human,train,en,3,twitter,0


In [ ]:
# Quick verification checks (dynamic row count after dropping failed generations)
label_counts = final_df["label"].value_counts().to_dict()
assert set(label_counts.keys()) == {0, 1}, f"Unexpected labels found: {label_counts}"
assert label_counts[0] == label_counts[1], f"Class imbalance detected: {label_counts}"

split_counts = final_df.groupby(["split", "label"]).size().unstack(fill_value=0)
display(split_counts)
display(final_df["split"].value_counts())
print(f"Final paired rows per class: {label_counts[0]}")
print(f"Total rows: {len(final_df)}")

label,0,1
split,,
test,400,400
train,1600,1600


,count
split,
train,3200
test,800


Final paired rows per class: 2000
Total rows: 4000


In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

drive_output_dir = Path('/content/drive/MyDrive/multisocial_outputs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
drive_output_path = drive_output_dir / OUTPUT_CSV_PATH

if 'final_df' in globals():
    final_df.to_csv(drive_output_path, index=False)
elif Path(OUTPUT_CSV_PATH).exists():
    shutil.copy2(OUTPUT_CSV_PATH, drive_output_path)
else:
    raise FileNotFoundError(
        f"Could not find {OUTPUT_CSV_PATH}. Run Cell 5 first to generate the dataset."
    )

print(f"Saved output to: {drive_output_path}")
print(f"File size: {drive_output_path.stat().st_size / 1024:.2f} KB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved output to: /content/drive/MyDrive/multisocial_outputs/multisocial_micro.csv
File size: 535.98 KB
